# 回収率の探索的分析（レース条件別）

`02_recovery_rate_analysis.ipynb` では「人気順に機械的に賭けたら回収率はどうなるか」を
人気（1番人気・2番人気…）だけで集計した。結果は「人気馬ほど回収率が高く、人気薄ほど低い」
（＝市場はおおむね効率的）というものだった。

このノートブックでは同じ「人気別の回収率」を、**レースの条件（コース種別・馬場状態・距離帯・出走頭数帯）で分けて**集計し直す。狙いは、全体をまとめると見えない「実は特定の条件下では歪みが大きい／小さい」というパターンを探すこと。

## 見方（統計の知識がなくても読めるように）

- 各表は「人気帯」×「条件」のマス目に回収率（%）を入れたヒートマップ。単勝・複勝など複数のパターンは1枚の図の中に横に並べて表示し、見比べやすくしている。
  - **100%が収支トントン**。それより高ければ得、低ければ損。
  - 色は **赤＝回収率が低い（損しやすい）、白＝100%付近（トントン）、青＝回収率が高い（得しやすい）**。
  - **色のスケールはノートブック全体で統一している**（同じ赤・青の濃さは、どの図でも同じ回収率を意味する）。なので違う図同士でも色の濃さをそのまま比べてよい。
- 各マスには `(n=件数)` も表示している。**件数が少ないマスの数字は信用しない**こと（サイコロを10回振って高い目が多く出ても「イカサマだ」とは言えないのと同じで、件数が少ないと「たまたま」の可能性が高い）。目安として n が数百以上あるマスだけ見る。件数が少ないマス（`MIN_CELL_N`未満）は文字を灰色にして目立たなくしている。
- 探しているのは「他の条件と比べて色が明らかに違うマス」。そこが次に深掘りする候補になる。

本実装（モデルを使った学習・EV計算）は `predictor/predictor/evaluation.py` 側にあるので、ここでの結果はあくまで仮説出し用。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm

from analysis.db import query_df
from analysis.plotting import setup_japanese_font

sns.set_style("whitegrid")
setup_japanese_font()  # sns.set_style がフォント設定を上書きするので、必ずこの後に呼ぶ
pd.set_option("display.max_columns", None)

# --- 人気帯(predictor/predictor/evaluation.py の _pop_tier と同じ区切り) ---
POP_TIER_ORDER = ["1番人気", "2-3番人気", "4-6番人気", "7番人気以下"]


def pop_tier(p: int) -> str:
    if p == 1:
        return "1番人気"
    elif p <= 3:
        return "2-3番人気"
    elif p <= 6:
        return "4-6番人気"
    else:
        return "7番人気以下"


# --- 距離帯 ---
DISTANCE_ORDER = ["短距離(~1400m)", "マイル(1401-1800m)", "中距離(1801-2400m)", "長距離(2401m~)"]


def distance_bucket(d: int) -> str:
    if d <= 1400:
        return "短距離(~1400m)"
    elif d <= 1800:
        return "マイル(1401-1800m)"
    elif d <= 2400:
        return "中距離(1801-2400m)"
    else:
        return "長距離(2401m~)"


# --- 出走頭数帯 ---
HEAD_COUNT_ORDER = ["少頭数(~9頭)", "中頭数(10-15頭)", "多頭数(16頭~)"]


def head_count_bucket(h: int) -> str:
    if h <= 9:
        return "少頭数(~9頭)"
    elif h <= 15:
        return "中頭数(10-15頭)"
    else:
        return "多頭数(16頭~)"


# --- 馬場状態（データ不備の値 "稍" "不" は除外する） ---
TRACK_CONDITION_ORDER = ["良", "稍重", "重", "不良"]


def parse_yen(series: pd.Series) -> pd.Series:
    """"1,310" のようなカンマ区切り文字列を float に変換する（欠損は 0 = 不的中扱い）。"""
    return pd.to_numeric(series.str.replace(",", "", regex=False), errors="coerce").fillna(0)


MIN_CELL_N = 30  # これ未満のマスはヒートマップ上で参考値扱い（薄く表示する）

# 回収率ヒートマップの色は、ノートブック全体で同じスケールに固定する
# （図ごとに色の基準が変わると、赤・青の濃さを図同士で比べられなくなるため）
# このノートブックで観測される回収率はだいたい40%〜100%の範囲に収まる想定。
RECOVERY_NORM = TwoSlopeNorm(vmin=40.0, vcenter=100.0, vmax=105.0)
RECOVERY_CMAP = "RdBu"


def recovery_pivot(
    df: pd.DataFrame, axis_col: str, axis_order: list[str], payout_col: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """人気帯 × axis_col で回収率(%)とサンプル数(n)のピボットテーブルを返す。"""
    g = df.groupby(["popularity_tier", axis_col], observed=True)
    n = g.size().unstack(axis_col).reindex(index=POP_TIER_ORDER, columns=axis_order)
    total_payout = g[payout_col].sum().unstack(axis_col).reindex(index=POP_TIER_ORDER, columns=axis_order)
    recovery = (total_payout / (n * 100) * 100).round(1)
    return recovery, n


def compute_prior_stats(
    mounts_df: pd.DataFrame, entity_col: str, date_col: str = "date"
) -> pd.DataFrame:
    """entity_col（騎手ID・父馬名など）ごとに「そのレースより前」までの累積成績を計算する。

    循環参照（同じレースの結果を使って対象を分類し、同じレースで回収率を測る）を防ぐための
    共通処理。mounts_df は最低限 [entity_col, date_col, "is_win"] を含む1行1騎乗のデータフレーム
    であること（race_id・horse_number など他の列があればそのまま残る）。

    同じ日に複数レースがある場合、当日の他のレース結果を「前情報」として使わないよう、
    日付単位で集計してから1日分ずらして累積する（同日内のレースの前後関係までは分からないため）。

    Returns
    -------
    元の mounts_df に prior_mounts（そのレースより前の累積騎乗数）・
    prior_win_rate（そのレースより前までの勝率）の列を追加したもの。
    """
    daily = (
        mounts_df.groupby([entity_col, date_col])["is_win"]
        .agg(daily_mounts="count", daily_wins="sum")
        .reset_index()
        .sort_values([entity_col, date_col])
    )
    grp = daily.groupby(entity_col)
    daily["prior_mounts"] = grp["daily_mounts"].cumsum() - daily["daily_mounts"]
    daily["prior_wins"] = grp["daily_wins"].cumsum() - daily["daily_wins"]
    daily["prior_win_rate"] = daily["prior_wins"] / daily["prior_mounts"]
    return mounts_df.merge(
        daily[[entity_col, date_col, "prior_mounts", "prior_win_rate"]],
        on=[entity_col, date_col],
        how="left",
    )


def tier_from_prior_rate(
    prior_win_rate: pd.Series,
    prior_mounts: pd.Series,
    min_mounts: int,
    low_max: float,
    high_min: float,
    order: list[str],
) -> pd.Series:
    """「そのレースより前」の勝率を、固定の閾値で下位/中位/上位に分類する。

    閾値（low_max, high_min）は全期間データから一度だけ求めた目安値を定数として渡す
    （対象ごとの「未来の成績」を見てその都度しきい値を決め直しているわけではないので、
    ここでは循環参照は起きない）。prior_mounts が min_mounts 未満の行（新人騎手・稀少血統など、
    まだ十分な実績が貯まっていない時点）は分類対象外（NaN）にする。
    """
    tier = pd.Series(pd.NA, index=prior_win_rate.index, dtype="object")
    qualified = prior_mounts >= min_mounts
    tier[qualified & (prior_win_rate <= low_max)] = order[0]
    tier[qualified & (prior_win_rate > low_max) & (prior_win_rate < high_min)] = order[1]
    tier[qualified & (prior_win_rate >= high_min)] = order[2]
    return tier


def _draw_recovery_heatmap(ax: plt.Axes, recovery: pd.DataFrame, n: pd.DataFrame, title: str):
    """1つの ax に回収率ヒートマップを描く（plot_recovery_grid の下請け関数）。

    列数が多いパネルでは、文字が隣のマスとぶつからないようにフォントサイズを自動で小さくする。
    """
    values = recovery.to_numpy(dtype=float)
    im = ax.imshow(values, cmap=RECOVERY_CMAP, norm=RECOVERY_NORM, aspect="auto")

    n_cols = len(recovery.columns)
    fontsize = 9 if n_cols <= 4 else (8 if n_cols <= 6 else 7)

    # 列ラベルが長い場合は斜めにして重なりを避ける
    max_label_len = max((len(str(c)) for c in recovery.columns), default=0)
    rotation = 20 if max_label_len > 4 else 0
    ax.set_xticks(range(len(recovery.columns)))
    ax.set_xticklabels(recovery.columns, rotation=rotation, ha="right" if rotation else "center", fontsize=fontsize)
    ax.set_yticks(range(len(recovery.index)))
    ax.set_yticklabels(recovery.index, fontsize=fontsize)
    ax.grid(False)

    for i in range(recovery.shape[0]):
        for j in range(recovery.shape[1]):
            val = recovery.iloc[i, j]
            cnt = n.iloc[i, j]
            if pd.isna(val) or pd.isna(cnt):
                ax.text(j, i, "n/a", ha="center", va="center", fontsize=fontsize - 1, color="gray")
                continue
            low_n = cnt < MIN_CELL_N
            label = f"{val:.1f}%\n(n={int(cnt):,})"
            text_color = "gray" if low_n else ("black" if 55 < val < 145 else "white")
            ax.text(j, i, label, ha="center", va="center", fontsize=fontsize, color=text_color, fontweight="bold")

    ax.set_title(title, fontsize=11)
    return im


def plot_recovery_grid(
    panels: list[tuple[str, pd.DataFrame, pd.DataFrame]], suptitle: str, ncols: int = 2
) -> None:
    """複数の回収率ヒートマップを1枚のグリッドにまとめて描画する（色スケールは全パネル共通）。

    Parameters
    ----------
    panels : list of (パネルのタイトル, recovery, n)
    suptitle : 図全体のタイトル
    ncols : 1行に並べるパネル数
    """
    n_panels = len(panels)
    ncols = min(ncols, n_panels)
    nrows = -(-n_panels // ncols)

    # パネルの列数（人気帯以外の軸のカテゴリ数）が多いほど、1パネルの横幅を広げる
    max_data_cols = max(len(recovery.columns) for _, recovery, _ in panels)
    panel_width = max(4.6, 0.95 * max_data_cols + 1.8)

    fig, axes = plt.subplots(
        nrows, ncols, figsize=(panel_width * ncols, 4.2 * nrows + 1), squeeze=False
    )

    im = None
    for idx, (title, recovery, n) in enumerate(panels):
        ax = axes[idx // ncols][idx % ncols]
        im = _draw_recovery_heatmap(ax, recovery, n, title)
    for idx in range(n_panels, nrows * ncols):
        axes[idx // ncols][idx % ncols].axis("off")

    fig.suptitle(suptitle, fontsize=14, fontweight="bold")
    fig.subplots_adjust(hspace=0.55, wspace=0.35, top=0.90 if nrows == 1 else 0.92)
    fig.colorbar(
        im,
        ax=axes,
        label="回収率 (%) ／ 100%が収支トントン・全パネル共通スケール",
        shrink=0.85,
        pad=0.02,
    )
    plt.show()

## データ読み込み

`race_results`（各馬の人気・馬番）に `races`（コース種別・距離・馬場状態・頭数）を結合し、`payoffs` から単勝・複勝の払戻金額を引く。1行 = 1レースの1頭。

なお `venue`（開催場）は東京・中山・京都・中京の中央4場以外はレース数が数十〜数百件しかなく（地方競馬の混在やスクレイピングの網羅範囲の偏りが原因）、条件別に割ると信頼できる集計にならないため、今回の軸からは除外した。

In [ ]:
base_df = query_df(
    """
    SELECT
        r.race_id,
        r.horse_number,
        r.popularity,
        ra.course_type,
        ra.distance,
        ra.track_condition,
        ra.head_count,
        win_p.payout AS win_payout,
        place_p.payout AS place_payout
    FROM race_results r
    JOIN races ra ON ra.race_id = r.race_id
    LEFT JOIN payoffs win_p
        ON win_p.race_id = r.race_id
        AND win_p.bet_type = '単勝'
        AND win_p.combination = r.horse_number
    LEFT JOIN payoffs place_p
        ON place_p.race_id = r.race_id
        AND place_p.bet_type = '複勝'
        AND place_p.combination = r.horse_number
    WHERE r.popularity IS NOT NULL AND r.popularity ~ '^[0-9]+$'
    """
)

base_df["popularity"] = base_df["popularity"].astype(int)
base_df["popularity_tier"] = base_df["popularity"].map(pop_tier)
base_df["win_payout"] = parse_yen(base_df["win_payout"])
base_df["place_payout"] = parse_yen(base_df["place_payout"])

base_df["distance_bucket"] = base_df["distance"].map(distance_bucket)
base_df["head_count_bucket"] = base_df["head_count"].map(head_count_bucket)

# データ不備（course_type が NaN、track_condition が想定外の値）の行は除外
base_df = base_df[base_df["course_type"].isin(["芝", "ダート"])]
base_df = base_df[base_df["track_condition"].isin(TRACK_CONDITION_ORDER)]

print(f"対象行数: {len(base_df):,}")
base_df.head()

## 軸1: コース種別（芝 / ダート）

芝レースとダートレースで、人気帯ごとの回収率の傾向に差があるかを見る。

In [ ]:
win_recovery, win_n = recovery_pivot(base_df, "course_type", ["芝", "ダート"], "win_payout")
place_recovery, place_n = recovery_pivot(base_df, "course_type", ["芝", "ダート"], "place_payout")

plot_recovery_grid(
    [
        ("単勝", win_recovery, win_n),
        ("複勝", place_recovery, place_n),
    ],
    "コース種別 × 人気帯の回収率",
    ncols=2,
)

## 軸2: 馬場状態（良 / 稍重 / 重 / 不良）

雨で馬場が悪くなるほど、人気（＝オッズ）と実際の強さの対応関係が崩れやすい、という仮説を確認する。

In [ ]:
win_recovery, win_n = recovery_pivot(base_df, "track_condition", TRACK_CONDITION_ORDER, "win_payout")
place_recovery, place_n = recovery_pivot(base_df, "track_condition", TRACK_CONDITION_ORDER, "place_payout")

plot_recovery_grid(
    [
        ("単勝", win_recovery, win_n),
        ("複勝", place_recovery, place_n),
    ],
    "馬場状態 × 人気帯の回収率",
    ncols=2,
)

## 軸3: 距離帯（短距離 / マイル / 中距離 / 長距離）

距離が長くなるほどレース展開の不確実性が増し、市場（オッズ）が読み違えやすくなる、という仮説を確認する。

In [ ]:
win_recovery, win_n = recovery_pivot(base_df, "distance_bucket", DISTANCE_ORDER, "win_payout")
place_recovery, place_n = recovery_pivot(base_df, "distance_bucket", DISTANCE_ORDER, "place_payout")

plot_recovery_grid(
    [
        ("単勝", win_recovery, win_n),
        ("複勝", place_recovery, place_n),
    ],
    "距離帯 × 人気帯の回収率",
    ncols=2,
)

## 軸4: 出走頭数帯（少頭数 / 中頭数 / 多頭数）

頭数が多いレースほど組み合わせが増え、市場が全馬を正確に評価しきれず歪みが残りやすい、という仮説を確認する。

In [ ]:
win_recovery, win_n = recovery_pivot(base_df, "head_count_bucket", HEAD_COUNT_ORDER, "win_payout")
place_recovery, place_n = recovery_pivot(base_df, "head_count_bucket", HEAD_COUNT_ORDER, "place_payout")

plot_recovery_grid(
    [
        ("単勝", win_recovery, win_n),
        ("複勝", place_recovery, place_n),
    ],
    "出走頭数帯 × 人気帯の回収率",
    ncols=2,
)

## 軸5（多変量）: 良馬場限定 × 距離帯 × オッズの分散 × 人気帯

ここまでは「条件1つ × 人気帯」だったが、ここでは条件を組み合わせる。

- **良馬場に絞る**（馬場状態による歪みの影響を固定して排除するため）。
- **「オッズの分散」**という新しい軸を追加する。これは1レースに出走する全馬の単勝オッズが、どれくらいバラついているかを表す指標。
  - 素のオッズ（1.5倍〜80倍など）をそのまま使うと大穴の数字だけが桁違いに大きくて指標が引っ張られるので、`log(オッズ)` に変換してからバラつき（標準偏差）を計算する。
  - 値が大きい＝本命と穴馬の評価差がはっきりしたレース、小さい＝みんな似たようなオッズで横並びの混戦。
  - レースごとに1つの値になるので、良馬場のレース全体を3等分（混戦／普通／本命はっきり）してグループ分けする。
- そのうえで、**距離帯ごとに**「オッズの分散」×「人気帯」の回収率ヒートマップを見る。

見たいのは「混戦レースだと人気薄の回収率が変わるか」「本命がはっきりしたレースだと1番人気の回収率がさらに上がるか」といったパターンで、特に前回見つかった「長距離×人気薄」がどちらのタイプのレースで起きているのかを確認する。

In [ ]:
good_track_df = query_df(
    """
    SELECT
        r.race_id,
        r.horse_number,
        r.popularity,
        r.odds,
        ra.course_type,
        ra.distance,
        ra.head_count,
        win_p.payout AS win_payout,
        place_p.payout AS place_payout
    FROM race_results r
    JOIN races ra ON ra.race_id = r.race_id
    LEFT JOIN payoffs win_p
        ON win_p.race_id = r.race_id
        AND win_p.bet_type = '単勝'
        AND win_p.combination = r.horse_number
    LEFT JOIN payoffs place_p
        ON place_p.race_id = r.race_id
        AND place_p.bet_type = '複勝'
        AND place_p.combination = r.horse_number
    WHERE r.popularity IS NOT NULL AND r.popularity ~ '^[0-9]+$'
      AND ra.track_condition = '良'
    """
)

good_track_df["odds_num"] = pd.to_numeric(good_track_df["odds"], errors="coerce")
good_track_df = good_track_df[good_track_df["odds_num"] > 0]
good_track_df = good_track_df[good_track_df["course_type"].isin(["芝", "ダート"])]

# レースごとに log(オッズ) の標準偏差を計算し、全体を3等分してグループ分けする
DISPERSION_ORDER = ["混戦(オッズ横並び)", "普通", "本命はっきり(差が大きい)"]

race_log_odds_std = good_track_df.groupby("race_id")["odds_num"].apply(lambda s: np.log(s).std())
race_dispersion_tier = pd.qcut(race_log_odds_std, 3, labels=DISPERSION_ORDER)
good_track_df["odds_dispersion_tier"] = good_track_df["race_id"].map(race_dispersion_tier)

good_track_df["popularity"] = good_track_df["popularity"].astype(int)
good_track_df["popularity_tier"] = good_track_df["popularity"].map(pop_tier)
good_track_df["distance_bucket"] = good_track_df["distance"].map(distance_bucket)
good_track_df["win_payout"] = parse_yen(good_track_df["win_payout"])
good_track_df["place_payout"] = parse_yen(good_track_df["place_payout"])

print(f"対象行数（良馬場）: {len(good_track_df):,}")
print(f"対象レース数: {good_track_df['race_id'].nunique():,}")
good_track_df[["odds_dispersion_tier"]].value_counts()

In [ ]:
# 距離帯ごとに「オッズ分散 × 人気帯」の回収率を、単勝・複勝まとめて1枚のグリッドで見る
panels = []
for bet_label, payout_col in [("単勝", "win_payout"), ("複勝", "place_payout")]:
    for db in DISTANCE_ORDER:
        sub = good_track_df[good_track_df["distance_bucket"] == db]
        recovery, n = recovery_pivot(sub, "odds_dispersion_tier", DISPERSION_ORDER, payout_col)
        panels.append((f"{bet_label}: {db}", recovery, n))

plot_recovery_grid(panels, "良馬場: 距離帯 × オッズ分散 × 人気帯の回収率", ncols=4)

In [ ]:
# 上のヒートマップだけだと細かい数値が読みにくいので、正確な数値を表でも確認できるようにしておく
rows = []
for db in DISTANCE_ORDER:
    sub = good_track_df[good_track_df["distance_bucket"] == db]
    win_recovery, win_n = recovery_pivot(sub, "odds_dispersion_tier", DISPERSION_ORDER, "win_payout")
    place_recovery, place_n = recovery_pivot(sub, "odds_dispersion_tier", DISPERSION_ORDER, "place_payout")
    for tier in DISPERSION_ORDER:
        for pt in POP_TIER_ORDER:
            rows.append(
                {
                    "距離帯": db,
                    "オッズ分散": tier,
                    "人気帯": pt,
                    "n": win_n.loc[pt, tier],
                    "単勝回収率": win_recovery.loc[pt, tier],
                    "複勝回収率": place_recovery.loc[pt, tier],
                }
            )

summary = pd.DataFrame(rows)
summary = summary[summary["n"] >= MIN_CELL_N]  # 参考にならないほど件数が少ないマスは除く
summary.sort_values(["距離帯", "人気帯", "オッズ分散"]).reset_index(drop=True)

## 軸6: グレード（重賞クラス）

`races.grade` は空欄しか入っていないので、代わりに `race_name`（例:「第56回日経賞(GII)」「3歳未勝利」）の末尾から `(GI)` `(GII)` `(GIII)` `(L)` を正規表現で拾い、それ以外（オープン・条件戦・未勝利・新馬など）は「平場」としてまとめる。これは`predictor/predictor/evaluation.py`の`evaluate_by_grade`と同じ区切り方。

**仮説**: G1のような大きなレースはプロの資金・情報が集まって市場が「賢く」なりやすく、平場（下級条件戦）の方が歪みが残りやすいのではないか。

In [ ]:
GRADE_ORDER = ["平場", "L", "GIII", "GII", "GI"]


def extract_grade(race_name) -> str:
    if not isinstance(race_name, str):
        return "平場"
    m = re.search(r"\((GIII|GII|GI|L)\)\s*$", race_name)
    return m.group(1) if m else "平場"


race_grade_df = query_df("SELECT race_id, race_name FROM races")
race_grade_df["grade_tier"] = race_grade_df["race_name"].map(extract_grade)
base_df = base_df.merge(race_grade_df[["race_id", "grade_tier"]], on="race_id", how="left")

print(base_df["grade_tier"].value_counts().reindex(GRADE_ORDER))

win_recovery, win_n = recovery_pivot(base_df, "grade_tier", GRADE_ORDER, "win_payout")
place_recovery, place_n = recovery_pivot(base_df, "grade_tier", GRADE_ORDER, "place_payout")

plot_recovery_grid(
    [
        ("単勝", win_recovery, win_n),
        ("複勝", place_recovery, place_n),
    ],
    "グレード × 人気帯の回収率",
    ncols=2,
)

## 軸7: 枠番（内枠 / 外枠）

競馬では「内枠（1〜2枠）が有利、外枠（7〜8枠）が不利」といったコース形状に由来する有利不利があるとよく言われる。もしこれがオッズにきちんと織り込まれていなければ、枠番によって回収率に差が出るはず。

**仮説**: 枠番による有利不利がオッズに完全には反映されておらず、特定の枠番で回収率が高い/低いパターンが出るのではないか。

In [ ]:
BRACKET_ORDER = list(range(1, 9))  # 1枠(内) 〜 8枠(外)

bracket_df = query_df(
    """
    SELECT race_id, horse_number, bracket_number
    FROM race_results
    WHERE bracket_number IS NOT NULL AND bracket_number ~ '^[0-9]+$'
    """
)
bracket_df["bracket_number"] = bracket_df["bracket_number"].astype(int)
base_df = base_df.merge(bracket_df, on=["race_id", "horse_number"], how="left")

win_recovery, win_n = recovery_pivot(base_df, "bracket_number", BRACKET_ORDER, "win_payout")
place_recovery, place_n = recovery_pivot(base_df, "bracket_number", BRACKET_ORDER, "place_payout")

plot_recovery_grid(
    [
        ("単勝", win_recovery, win_n),
        ("複勝", place_recovery, place_n),
    ],
    "枠番 × 人気帯の回収率（1=内枠 〜 8=外枠）",
    ncols=2,
)

## 軸8: 騎手の勝率帯（「有名騎手は過大評価されて買われすぎる」仮説）

騎手ごとに、このデータ全体での通算勝率（勝った数 ÷ 騎乗数）を計算し、騎乗数が500以上ある騎手だけを対象に「勝率が高い順」で3等分（下位／中位／上位）する。勝率が高い騎手＝いわゆる「上手い・有名な騎手」の目安として使う。

**仮説**: 有名な騎手が乗っていると、その馬の実力以上に人気になりやすい（＝過大評価されて回収率が下がりやすい）のではないか。

**注意（重要）**: ここでの「勝率」はこのデータ全体（同じ期間・同じレース群）から計算している。つまり「その騎手がこのデータの中でたくさん勝ってきた実績」を使って騎手を分類し、その同じデータで回収率を見ている ―― 多少循環的な集計になっている。将来の予測に使えるかどうかの検証ではなく、「過去の傾向として、勝率が高い騎手ほど回収率が低い/高いか」を確認する記述的な分析として見てほしい。

In [ ]:
MIN_MOUNTS = 500  # これ未満の騎乗数の騎手は勝率が不安定なので集計対象から外す
JOCKEY_TIER_ORDER = ["下位(勝率)", "中位(勝率)", "上位(勝率)"]

jockey_stats = query_df(
    """
    SELECT jockey_id,
           count(*) AS mounts,
           sum(case when finishing_position = '1' then 1 else 0 end) AS wins
    FROM race_results
    WHERE jockey_id IS NOT NULL AND jockey_id <> ''
    GROUP BY jockey_id
    """
)
jockey_stats["win_rate"] = jockey_stats["wins"] / jockey_stats["mounts"]
qualified_jockeys = jockey_stats[jockey_stats["mounts"] >= MIN_MOUNTS].copy()
qualified_jockeys["jockey_tier"] = pd.qcut(qualified_jockeys["win_rate"], 3, labels=JOCKEY_TIER_ORDER)

print(f"対象騎手数（騎乗数{MIN_MOUNTS}以上）: {len(qualified_jockeys):,} / 全騎手 {len(jockey_stats):,}")

jockey_df = query_df("SELECT race_id, horse_number, jockey_id FROM race_results")
base_df = base_df.merge(jockey_df, on=["race_id", "horse_number"], how="left")
base_df = base_df.merge(qualified_jockeys[["jockey_id", "jockey_tier"]], on="jockey_id", how="left")
# 騎乗数が MIN_MOUNTS 未満の騎手（新人・稀少騎手）は jockey_tier が欠損 -> pivot で自動的に集計対象外になる

win_recovery, win_n = recovery_pivot(base_df, "jockey_tier", JOCKEY_TIER_ORDER, "win_payout")
place_recovery, place_n = recovery_pivot(base_df, "jockey_tier", JOCKEY_TIER_ORDER, "place_payout")

plot_recovery_grid(
    [
        ("単勝", win_recovery, win_n),
        ("複勝", place_recovery, place_n),
    ],
    "騎手の通算勝率帯 × 人気帯の回収率",
    ncols=2,
)

## 軸9: 血統（父馬の産駒勝率）

騎手と同じやり方で、父馬（`horses.sire`）ごとに産駒の通算勝率を計算し、産駒の出走数が500以上ある父馬だけを対象に3等分（下位／中位／上位）する。良い父馬の産駒＝素質の高い血統、の目安として使う。

**仮説**: 良血統の産駒が人気薄に沈んでいるとき、市場は血統的な素質を十分に評価できておらず、回収率が高くなるのではないか。騎手のときと違い、血統は「生まれつきの資質」で毎レース変わらないため、市場が学習しやすいはずだが、地味な血統情報まで完全に反映されているかは分からない。

**注意**: 騎手のときと同じ「勝率の算出にこのデータ全体の結果を使っている」循環性の注意点がここでも当てはまる。

In [ ]:
MIN_SIRE_MOUNTS = 500  # これ未満の産駒出走数の父馬は勝率が不安定なので集計対象から外す
SIRE_TIER_ORDER = ["下位(勝率)", "中位(勝率)", "上位(勝率)"]

sire_stats = query_df(
    """
    SELECT h.sire,
           count(*) AS mounts,
           sum(case when r.finishing_position = '1' then 1 else 0 end) AS wins
    FROM race_results r
    JOIN horses h ON h.horse_id = r.horse_id
    WHERE h.sire IS NOT NULL AND h.sire <> ''
    GROUP BY h.sire
    """
)
sire_stats["win_rate"] = sire_stats["wins"] / sire_stats["mounts"]
qualified_sires = sire_stats[sire_stats["mounts"] >= MIN_SIRE_MOUNTS].copy()
qualified_sires["sire_tier"] = pd.qcut(qualified_sires["win_rate"], 3, labels=SIRE_TIER_ORDER)

print(f"対象父馬数（産駒出走数{MIN_SIRE_MOUNTS}以上）: {len(qualified_sires):,} / 全父馬 {len(sire_stats):,}")

sire_df = query_df(
    """
    SELECT r.race_id, r.horse_number, h.sire
    FROM race_results r
    JOIN horses h ON h.horse_id = r.horse_id
    WHERE h.sire IS NOT NULL AND h.sire <> ''
    """
)
base_df = base_df.merge(sire_df, on=["race_id", "horse_number"], how="left")
base_df = base_df.merge(qualified_sires[["sire", "sire_tier"]], on="sire", how="left")
# 産駒出走数が MIN_SIRE_MOUNTS 未満の父馬は sire_tier が欠損 -> pivot で自動的に集計対象外になる

win_recovery, win_n = recovery_pivot(base_df, "sire_tier", SIRE_TIER_ORDER, "win_payout")
place_recovery, place_n = recovery_pivot(base_df, "sire_tier", SIRE_TIER_ORDER, "place_payout")

plot_recovery_grid(
    [
        ("単勝", win_recovery, win_n),
        ("複勝", place_recovery, place_n),
    ],
    "父馬の産駒勝率帯 × 人気帯の回収率",
    ncols=2,
)

## 軸10: 騎手の勝率帯（循環性を除いた再検証）

軸8では「このデータ全体の勝率」で騎手を分類していたため、循環している（そのレースの結果も含めて騎手を評価し、同じレースで回収率を測っている）という指摘があった。ここでは共通処理 `compute_prior_stats` / `tier_from_prior_rate`（上のセルアップ部分に定義）を使い、**各レースより前の時点までの騎乗成績だけ**で騎手を分類し直す。

- 騎乗数・勝率は日付単位で累積し、「そのレース当日より前」の実績のみを使う（同日の他レースの結果は使わない）。
- 分類のしきい値（勝率何%以上を「上位」とするか）は、全期間データから一度だけ求めた目安値（下位/中位の境目 = 4%、中位/上位の境目 = 6%）を固定で使う。個々の騎手の将来の成績で境目を決め直しているわけではないので、ここは循環参照にならない。
- 累積騎乗数が500件に満たない時点（新人騎手のデビュー直後など）は分類対象外にする。

**軸8（循環あり）の結果**: 7番人気以下で 下位53.8% → 中位65.4% → 上位73.1%（単勝）と19ptの差。この差が、循環性を除いても残るかを確認する。

In [ ]:
JOCKEY_LOW_MAX = 0.04  # 下位/中位の境目（全期間データの3分位点から丸めた固定値）
JOCKEY_HIGH_MIN = 0.06  # 中位/上位の境目

jockey_mounts_df = query_df(
    """
    SELECT
        r.race_id,
        r.horse_number,
        r.jockey_id AS entity,
        ra.date,
        case when r.finishing_position = '1' then 1 else 0 end AS is_win
    FROM race_results r
    JOIN races ra ON ra.race_id = r.race_id
    WHERE r.jockey_id IS NOT NULL AND r.jockey_id <> ''
    """
)
jockey_mounts_df = compute_prior_stats(jockey_mounts_df, "entity")
jockey_mounts_df["jockey_tier_prior"] = tier_from_prior_rate(
    jockey_mounts_df["prior_win_rate"],
    jockey_mounts_df["prior_mounts"],
    MIN_MOUNTS,
    JOCKEY_LOW_MAX,
    JOCKEY_HIGH_MIN,
    JOCKEY_TIER_ORDER,
)

base_df = base_df.merge(
    jockey_mounts_df[["race_id", "horse_number", "jockey_tier_prior"]],
    on=["race_id", "horse_number"],
    how="left",
)

win_recovery, win_n = recovery_pivot(base_df, "jockey_tier_prior", JOCKEY_TIER_ORDER, "win_payout")
place_recovery, place_n = recovery_pivot(base_df, "jockey_tier_prior", JOCKEY_TIER_ORDER, "place_payout")

plot_recovery_grid(
    [
        ("単勝", win_recovery, win_n),
        ("複勝", place_recovery, place_n),
    ],
    "騎手の「そのレースより前」の勝率帯 × 人気帯の回収率（循環性を除いた版）",
    ncols=2,
)

## 軸11: 父馬の産駒勝率帯（循環性を除いた再検証）

軸9と同じ問題（勝率の算出にこのデータ全体の結果を使っている）を、騎手と同じ仕組み（`compute_prior_stats` / `tier_from_prior_rate`）で修正する。**各レースより前の時点までの産駒成績だけ**で父馬を分類し直す。

- しきい値（下位/中位の境目 = 6%、中位/上位の境目 = 8%）も全期間データから一度だけ求めた固定値。
- 累積産駒出走数が500件に満たない時点（供用され始めたばかりの父馬など）は分類対象外。

**軸9（循環あり）の結果**: 1番人気〜7番人気以下の全ての人気帯で、上位血統が下位血統より10〜14pt回収率が高かった（単勝）。血統は「毎レース変わらない資質」なので、循環性の影響は騎手より小さいと予想しているが、実際にどう変わるかを確認する。

In [ ]:
SIRE_LOW_MAX = 0.06  # 下位/中位の境目（全期間データの3分位点から丸めた固定値）
SIRE_HIGH_MIN = 0.08  # 中位/上位の境目

sire_mounts_df = query_df(
    """
    SELECT
        r.race_id,
        r.horse_number,
        h.sire AS entity,
        ra.date,
        case when r.finishing_position = '1' then 1 else 0 end AS is_win
    FROM race_results r
    JOIN races ra ON ra.race_id = r.race_id
    JOIN horses h ON h.horse_id = r.horse_id
    WHERE h.sire IS NOT NULL AND h.sire <> ''
    """
)
sire_mounts_df = compute_prior_stats(sire_mounts_df, "entity")
sire_mounts_df["sire_tier_prior"] = tier_from_prior_rate(
    sire_mounts_df["prior_win_rate"],
    sire_mounts_df["prior_mounts"],
    MIN_SIRE_MOUNTS,
    SIRE_LOW_MAX,
    SIRE_HIGH_MIN,
    SIRE_TIER_ORDER,
)

base_df = base_df.merge(
    sire_mounts_df[["race_id", "horse_number", "sire_tier_prior"]],
    on=["race_id", "horse_number"],
    how="left",
)

win_recovery, win_n = recovery_pivot(base_df, "sire_tier_prior", SIRE_TIER_ORDER, "win_payout")
place_recovery, place_n = recovery_pivot(base_df, "sire_tier_prior", SIRE_TIER_ORDER, "place_payout")

plot_recovery_grid(
    [
        ("単勝", win_recovery, win_n),
        ("複勝", place_recovery, place_n),
    ],
    "父馬の「その時点までの」産駒勝率帯 × 人気帯の回収率（循環性を除いた版）",
    ncols=2,
)

## 軸12: 調教師の勝率帯（循環性を除いた版で最初から）

騎手・血統と同じ「エンティティの実績で分類する」パターンを、調教師（`race_results.trainer_id`）にも適用する。今回は最初から `compute_prior_stats` / `tier_from_prior_rate` を使い、各レースより前の実績だけで分類する（軸8・9のような循環ありの集計はやらない）。

しきい値は騎手・血統と同じやり方で全期間データから一度だけ求めた固定値（下位/中位の境目 = 5%、中位/上位の境目 = 8%）。累積騎乗数500件以上の調教師を対象にする（対象調教師数は370名、騎手の278名・父馬の146頭より層が厚い）。

**仮説**: 「仕上げの上手い調教師」の馬は、人気薄でも実際の走りが市場の評価を上回りやすいのではないか（騎手と同じロジック）。

In [ ]:
MIN_TRAINER_MOUNTS = 500
TRAINER_LOW_MAX = 0.05  # 下位/中位の境目（全期間データの3分位点から丸めた固定値）
TRAINER_HIGH_MIN = 0.08  # 中位/上位の境目
TRAINER_TIER_ORDER = ["下位(勝率)", "中位(勝率)", "上位(勝率)"]

trainer_mounts_df = query_df(
    """
    SELECT
        r.race_id,
        r.horse_number,
        r.trainer_id AS entity,
        ra.date,
        case when r.finishing_position = '1' then 1 else 0 end AS is_win
    FROM race_results r
    JOIN races ra ON ra.race_id = r.race_id
    WHERE r.trainer_id IS NOT NULL AND r.trainer_id <> ''
    """
)
trainer_mounts_df = compute_prior_stats(trainer_mounts_df, "entity")
trainer_mounts_df["trainer_tier_prior"] = tier_from_prior_rate(
    trainer_mounts_df["prior_win_rate"],
    trainer_mounts_df["prior_mounts"],
    MIN_TRAINER_MOUNTS,
    TRAINER_LOW_MAX,
    TRAINER_HIGH_MIN,
    TRAINER_TIER_ORDER,
)

base_df = base_df.merge(
    trainer_mounts_df[["race_id", "horse_number", "trainer_tier_prior"]],
    on=["race_id", "horse_number"],
    how="left",
)

win_recovery, win_n = recovery_pivot(base_df, "trainer_tier_prior", TRAINER_TIER_ORDER, "win_payout")
place_recovery, place_n = recovery_pivot(base_df, "trainer_tier_prior", TRAINER_TIER_ORDER, "place_payout")

plot_recovery_grid(
    [
        ("単勝", win_recovery, win_n),
        ("複勝", place_recovery, place_n),
    ],
    "調教師の「そのレースより前」の勝率帯 × 人気帯の回収率",
    ncols=2,
)

## 軸13: 馬体重の増減（レース当日発表の情報）

ここまではすべて「騎手・血統・調教師などエンティティの評判」を使った軸だったが、これはまったく違うタイプの情報。`race_results.horse_weight_diff`（前走からの馬体重増減、kg）は、レース当日の発表なので市場（オッズ）が確定する直前の情報であり、みんなが冷静に消化しきれていない可能性がある。

「馬体重が大きく増減すると調子落ちのサイン」という有名な俗説を検証する。増減±6kgを目安に3グループに分ける（データ全体で見ると、-6kg以下が約16%、+6kg以上が約18%、残りの66%が-5〜+5kgの「通常範囲」）。

**仮説**: 馬体重が大きく増減した馬（特に人気薄）は、市場がその情報を十分織り込めておらず、回収率に偏りが出るのではないか。

In [ ]:
WEIGHT_DIFF_ORDER = ["大幅減(-6kg以下)", "通常(-5~+5kg)", "大幅増(+6kg以上)"]


def weight_diff_bucket(d: float) -> str:
    if d <= -6:
        return "大幅減(-6kg以下)"
    elif d >= 6:
        return "大幅増(+6kg以上)"
    else:
        return "通常(-5~+5kg)"


weight_df = query_df(
    """
    SELECT race_id, horse_number, horse_weight_diff
    FROM race_results
    WHERE horse_weight_diff IS NOT NULL
    """
)
weight_df["weight_diff_bucket"] = weight_df["horse_weight_diff"].map(weight_diff_bucket)
base_df = base_df.merge(
    weight_df[["race_id", "horse_number", "weight_diff_bucket"]],
    on=["race_id", "horse_number"],
    how="left",
)

win_recovery, win_n = recovery_pivot(base_df, "weight_diff_bucket", WEIGHT_DIFF_ORDER, "win_payout")
place_recovery, place_n = recovery_pivot(base_df, "weight_diff_bucket", WEIGHT_DIFF_ORDER, "place_payout")

plot_recovery_grid(
    [
        ("単勝", win_recovery, win_n),
        ("複勝", place_recovery, place_n),
    ],
    "馬体重増減 × 人気帯の回収率",
    ncols=2,
)

## 軸14: 馬主の通算勝率帯（循環性を除いた版で最初から）

騎手・調教師・血統と同じ「エンティティの実績で分類する」パターンを、馬主（`race_results.owner`）にも適用する。最初から `compute_prior_stats` / `tier_from_prior_rate` を使い、各レースより前の実績だけで分類する（軸12の調教師と同じやり方）。

馬主は `horses.owner_id`（構造化ID）だと入力率が34%程度と低いため、代わりに入力率99.9%の `race_results.owner`（馬主名の文字列）をエンティティキーとして使う。表記ゆれで同一馬主が別エンティティとして分かれる可能性はあるが、`sire`/`dam`（同じく文字列キー）と同程度のリスクとして許容する。

しきい値は全期間データ（騎乗数500以上の馬主314名）の勝率3分位点から一度だけ求めた固定値（下位/中位の境目 = 6%、中位/上位の境目 = 8.5%）。累積出走数が500件に満たない時点は分類対象外。

**仮説**: 騎手・調教師は「レースでの技量」で市場が過小評価しがちな軸だったが、馬主は技量ではなく「資金力・馬の選定眼」を反映する軸。市場（一般の馬券購入者）は馬主情報をほとんど参照しないと言われるため、軽視されている可能性がある。

In [ ]:
MIN_OWNER_MOUNTS = 500
OWNER_LOW_MAX = 0.06  # 下位/中位の境目（500騎乗以上の馬主314名、勝率3分位点から丸めた固定値）
OWNER_HIGH_MIN = 0.085  # 中位/上位の境目
OWNER_TIER_ORDER = ["下位(勝率)", "中位(勝率)", "上位(勝率)"]

owner_mounts_df = query_df(
    """
    SELECT
        r.race_id,
        r.horse_number,
        r.owner AS entity,
        ra.date,
        case when r.finishing_position = '1' then 1 else 0 end AS is_win
    FROM race_results r
    JOIN races ra ON ra.race_id = r.race_id
    WHERE r.owner IS NOT NULL AND r.owner <> ''
    """
)
owner_mounts_df = compute_prior_stats(owner_mounts_df, "entity")
owner_mounts_df["owner_tier_prior"] = tier_from_prior_rate(
    owner_mounts_df["prior_win_rate"],
    owner_mounts_df["prior_mounts"],
    MIN_OWNER_MOUNTS,
    OWNER_LOW_MAX,
    OWNER_HIGH_MIN,
    OWNER_TIER_ORDER,
)

base_df = base_df.merge(
    owner_mounts_df[["race_id", "horse_number", "owner_tier_prior"]],
    on=["race_id", "horse_number"],
    how="left",
)

win_recovery, win_n = recovery_pivot(base_df, "owner_tier_prior", OWNER_TIER_ORDER, "win_payout")
place_recovery, place_n = recovery_pivot(base_df, "owner_tier_prior", OWNER_TIER_ORDER, "place_payout")

print("--- 単勝回収率(%) ---")
print(win_recovery)
print("\n--- 単勝 n ---")
print(win_n)
print("\n--- 複勝回収率(%) ---")
print(place_recovery)
print("\n--- 複勝 n ---")
print(place_n)

plot_recovery_grid(
    [
        ("単勝", win_recovery, win_n),
        ("複勝", place_recovery, place_n),
    ],
    "馬主の「そのレースより前」の勝率帯 × 人気帯の回収率",
    ncols=2,
)


## 軸15: 出走間隔（休み明け・連闘）× 人気帯

前走からの経過日数（同じ`horse_id`の前レース日付との差）で「連闘・休み明け」を分類し、人気帯別回収率を見る。馬は同日に複数レースを走れないため、騎手・調教師・馬主のときのような「同日集計してから1日ずらす」処理は不要で、`horse_id`ごとに日付でソートして単純に前レースとの差を取ればよい（デビュー戦など前走がない行はNaNになり自動的に集計対象外）。

これは「そのレースより前の実績」ではなく「その馬自身の前走日」という、対象馬自身の確定した過去の事実なので、循環参照のリスクはそもそもない。

In [ ]:
REST_DAYS_ORDER = ["連闘(~13日)", "中2-4週(14-34日)", "中5-8週(35-63日)", "中9-16週(64-119日)", "休み明け(120日~)"]


def rest_days_bucket(d: float) -> str | float:
    if pd.isna(d):
        return np.nan
    if d <= 13:
        return "連闘(~13日)"
    elif d <= 34:
        return "中2-4週(14-34日)"
    elif d <= 63:
        return "中5-8週(35-63日)"
    elif d <= 119:
        return "中9-16週(64-119日)"
    else:
        return "休み明け(120日~)"


horse_dates_df = query_df(
    """
    SELECT r.race_id, r.horse_number, r.horse_id, ra.date
    FROM race_results r
    JOIN races ra ON ra.race_id = r.race_id
    WHERE r.horse_id IS NOT NULL AND r.horse_id <> ''
    """
)
horse_dates_df["date"] = pd.to_datetime(horse_dates_df["date"], errors="coerce")
horse_dates_df = horse_dates_df.dropna(subset=["date"]).sort_values(["horse_id", "date"])
horse_dates_df["prev_date"] = horse_dates_df.groupby("horse_id")["date"].shift(1)
horse_dates_df["rest_days"] = (horse_dates_df["date"] - horse_dates_df["prev_date"]).dt.days
horse_dates_df["rest_days_bucket"] = horse_dates_df["rest_days"].map(rest_days_bucket)

print(f"前走ありの行: {horse_dates_df['rest_days'].notna().sum():,} / {len(horse_dates_df):,}")
print(horse_dates_df["rest_days_bucket"].value_counts().reindex(REST_DAYS_ORDER))

base_df = base_df.merge(
    horse_dates_df[["race_id", "horse_number", "rest_days_bucket"]],
    on=["race_id", "horse_number"],
    how="left",
)

win_recovery, win_n = recovery_pivot(base_df, "rest_days_bucket", REST_DAYS_ORDER, "win_payout")
place_recovery, place_n = recovery_pivot(base_df, "rest_days_bucket", REST_DAYS_ORDER, "place_payout")

print("\n--- 単勝回収率(%) ---")
print(win_recovery)
print("\n--- 単勝 n ---")
print(win_n)
print("\n--- 複勝回収率(%) ---")
print(place_recovery)
print("\n--- 複勝 n ---")
print(place_n)

plot_recovery_grid(
    [
        ("単勝", win_recovery, win_n),
        ("複勝", place_recovery, place_n),
    ],
    "出走間隔（休み明け・連闘） × 人気帯の回収率",
    ncols=2,
)


## メモ

対象 781,177 行（芝・ダートのみ、馬場状態が正常値の行のみ）で集計した結果。

- **コース種別（芝 / ダート）**: 人気帯ごとの回収率はほぼ同じ（差は1〜2pt程度）。ここに歪みはなさそう。
- **馬場状態（良→不良）**: 馬場が悪くなるほど「1番人気の回収率が下がり（77.4%→72.5%）、7番人気以下の回収率が上がる（63.3%→70.3%、複勝は66.3%→73.6%）」という一貫した傾向が見えた。雨で馬場が悪化すると、人気（＝オッズ）と実際の強さの対応が崩れやすい、という仮説を裏付ける結果。
- **距離帯**: **長距離（2401m以上）の7番人気以下**は、単勝77.2%・複勝72.8%と、他の距離帯（短距離〜中距離は62〜67%）より10pt以上高い。2-3番人気も長距離で83.6%と他より高め。長距離だけデータ件数が少なめ（7番人気以下でn=8,550、他は6〜19万件）だが、単勝・複勝の両方で同じ方向に出ているため偶然の可能性は低そう。
- **出走頭数帯**: 少頭数（〜9頭）は1番人気の回収率が高め（単勝79.7%・複勝85.8%）、多頭数（16頭〜）は7番人気以下がやや高め（単勝66.0%・複勝67.4%）。方向感はあるが差は5pt程度で、距離帯ほど明確ではない。

### 軸5（多変量: 良馬場×距離帯×オッズ分散）でわかったこと

ここが今回で一番はっきりした結果。良馬場に絞り、距離帯ごとに「オッズ分散（混戦度合い）× 人気帯」で回収率を見ると、**短距離・マイル・中距離の3つの距離帯すべてで同じパターン**が出た。

- **7番人気以下（人気薄）の回収率は、レースが「本命はっきり」なほど大きく下がる**。
  - 短距離: 混戦78.2% → 普通61.7% → 本命はっきり49.2%（単勝）
  - マイル: 混戦72.9% → 普通66.9% → 本命はっきり54.2%（単勝）
  - 中距離: 混戦67.7% → 普通68.9% → 本命はっきり47.6%（単勝）
  - 複勝でもほぼ同じ傾向。件数は各マス数千〜数万件あり信頼できる。
- 一方で **1番人気（本命）の回収率は、オッズ分散との関係がはっきりしない**（短距離ではやや上がる、中距離ではやや下がる、など方向感がバラバラ）。
- **長距離（2401m以上）は良馬場×オッズ分散で切ると件数が130〜3,657件まで減ってしまい**、特に「普通」「本命はっきり」の集計は信頼できない（極端な値が出ている）。「混戦」だけは件数がまずまず（527〜3,657件）で、7番人気以下は単勝74.5%・複勝80.1%と、短距離〜中距離の混戦グループより高め＝**「長距離×人気薄」の歪みは、その中でも特に混戦レースで強く出ている可能性がある**（ただし長距離全体でのn=8,550よりさらに絞られるため確定的ではない）。

**まとめると**: 「レースが横並びの混戦かどうか」は、距離帯によらず人気薄の回収率を大きく左右する、これまでで一番はっきりした軸。**「本命がはっきりしたレースの人気薄には手を出さない」**というだけでも、7番人気以下の平均的な負けを大きく減らせそう。

### 軸6〜7（グレード・枠番）でわかったこと

- **グレード**: `races.grade`が空だったので`race_name`末尾の`(GI)`等から抽出。平場（77.5/80.1/81.0/64.2%）とGI（82.3/80.3/78.3/64.4%）はほぼ同水準で、「G1は市場が賢くなる」という仮説ははっきり裏付けられなかった。ただし**GII・GIIIの7番人気以下だけ回収率が高め**（GIII 69.9%・n=10,037、GII 79.0%・n=6,191、平場64.2%と比べて明確に高い）で、中堅グレードのレースで人気薄がやや過小評価されている可能性がある。Lグレードは件数が小さく（1番人気n=302など）参考程度。**GIの1番人気は82.3%とどのグレードよりも高く**、「本当に抜けた本命がいるGIレースでは、市場は本命側をやや過小評価している」という見方もできる。
- **枠番**: 1〜8枠でほとんど差がなく、明確な内枠/外枠バイアスは出なかった（7番人気以下は60〜69%の範囲で上下）。件数は各マス数万件と十分あるので、単純平均では「枠番だけ」で回収率に効くパターンはなさそう（内枠有利・外枠不利はコース・距離ごとに向きが変わるため、全体平均すると打ち消し合っている可能性が高い）。

### 軸8〜11（騎手・血統: 循環性ありvs循環性を除いた版の比較）

軸8・9では「このデータ全体の勝率」で騎手・父馬を分類していたため、**同じレースの結果を使って対象を分類し、同じレースで回収率を測る循環参照になっていた**（ユーザー指摘）。そこで `compute_prior_stats` / `tier_from_prior_rate` という再利用可能な仕組みを実装し、**各レースより前の時点までの実績だけ**で分類し直した（軸10・11）。しきい値は全期間データから一度だけ求めた固定値を使うので、対象ごとの将来の成績で境目を決め直す循環は起きない。

- **騎手（軸8→軸10）: 効果はほぼそのまま残った。** 7番人気以下の単勝回収率は、循環あり版で 下位53.8%→中位65.4%→上位73.1%（19.3pt差）、循環を除いた版で **下位55.0%→中位67.5%→上位71.6%（16.6pt差）**とほぼ同じ大きさ。件数も76,088/100,551/152,511と十分大きい。**「勝率の高い騎手が人気薄の馬に乗っているときは回収率が高い」というのは、循環参照のせいではなく、実際に残る傾向らしい**。ただし前述の通り最高でも71.6%（100%未満）なので、これ単体で黒字化には届かない。
- **血統（軸9→軸11）: 効果は残るが、循環あり版よりノイズが増えて崩れた。** 循環あり版は1番人気〜7番人気以下の全人気帯で上位血統が下位血統より10〜14pt高い、というきれいな一貫パターンだったが、循環を除いた版では「下位→上位」の差自体はどの人気帯でも残っている（+4〜+14pt）ものの、**中位タイヤが下位・上位より低くなる逆転が複数の人気帯で発生**（4-6番人気: 68.6%→85.4%→82.6%、7番人気以下: 66.8%→60.6%→73.0%）。累積産駒数500件という条件を満たすまで対象外にする都合上、1番人気の下位タイヤはn=500と小さく、参考程度。**血統の効果は「ある」とは言えそうだが、騎手ほどきれいでも安定でもない**、というのが正直な評価。

### 軸12〜13（調教師・馬体重増減）でわかったこと

- **調教師の勝率帯（今回で最大の効果、循環性なしで最初から検証）**: 7番人気以下の単勝回収率が **下位50.6%→中位62.5%→上位74.9%と24.3ptもの差**。騎手（16.6pt差）よりもさらに大きい。件数も78,583/147,336/107,100と非常に大きく、対象調教師数も370名（騎手278名・父馬146頭より層が厚い）。他の人気帯（1番人気〜4-6番人気）ではこの差はほぼ見られず、**効果は人気薄に集中**というパターンは騎手と同じ。「仕上げの上手い調教師の馬は、人気薄でも実力を発揮しやすい」という見方を裏付ける、これまでで最も差の大きい歪み。ただし例によって最高でも74.9%（100%未満）。
- **馬体重の増減**: 「大幅な増減は調子落ちのサイン」という俗説は、回収率の観点ではほぼ裏付けられなかった。7番人気以下でも 大幅減61.3%・通常65.1%・大幅増65.6% とわずかな差しかなく、他の人気帯でも1〜3pt程度の差にとどまる。**むしろ大幅増のほうがわずかに回収率が高い**傾向すらあり、「大幅増＝悪い」という単純な俗説的な歪みは見つからなかった。馬体重は競馬新聞やレース直前の情報で広く周知されるため、市場がすでに適切に織り込んでいる可能性が高い。

### 次にやること（仮説の優先順位）

1. **調教師の勝率帯×人気薄**が今回で最も差が大きく、サンプルも豊富。次に検証・活用する価値が最も高い。
2. **騎手の勝率帯×人気薄**も循環性を除いても効果が残った、信頼できる歪み。調教師・騎手はおそらく相関する（良い調教師は良い騎手を起用しやすい）ので、両方を組み合わせた分析も価値がありそう。
3. **「本命がはっきりしたレースの人気薄を避ける」ルール**も手堅い。単純な除外条件として`predictor`のベット選択ロジックに組み込みやすい。
4. **父馬の産駒勝率**は方向感はあるが循環性を除くとノイズが増えた。サンプルを増やすか、母父・母馬など他の血統情報も試して再検証する価値がある。
5. **馬体重の増減は今回は空振り**。優先度は下げてよい。
6. **長距離×混戦×人気薄**が理論上一番良さそうな組み合わせだが、件数がまだ少ないので再検証が必要。
7. **馬場状態が悪いレースの穴馬**も同じ方向の歪みなので、不良馬場×混戦のような組み合わせも試す価値がある。
8. **枠番はこの粒度（全体平均）では効果なし**。優先度は下げてよさそう。
9. いずれも「まだ100%を超えてはいない」（最高でも複勝8割台）。この段階ではまだ回収率110%には届かないが、`predictor` のモデル予測・EVフィルタとこれらの条件を掛け合わせることで、歪みの大きい条件に絞ったベット選択の精度を上げられる可能性がある。特に調教師・騎手の勝率帯は、`predictor`のベット選択ロジックに直接組み込みやすい候補。